# QR-проверка подлинности

Ветка: **[ДПО ИИ] 10 — Сертификат или удостоверение**.

Этот ноутбук является частью будущей Jupyter Book-книги и предназначен для воспроизводимой демонстрации модели.

## 1. Назначение QR-проверки

QR-код нужен не для хранения всех данных сертификата, а для перехода на публичную страницу проверки. Публичная страница должна показывать минимальный набор сведений и не раскрывать служебные идентификаторы.

In [ ]:
from pathlib import Path
import secrets
import json
import pandas as pd
import qrcode
import matplotlib.pyplot as plt

BASE = Path.cwd()
verification_code = 'ver_' + secrets.token_urlsafe(18)
base_url = 'https://example.org/certificates/verify'
verification_url = f'{base_url}/{verification_code}'
verification_code, verification_url

## 2. Генерация QR-кода

В реальной системе QR-код должен формироваться из `verificationUrl`. Сам `verificationCode` не должен быть последовательным или угадываемым.

In [ ]:
qr = qrcode.QRCode(version=1, box_size=10, border=4)
qr.add_data(verification_url)
qr.make(fit=True)
img = qr.make_image(fill_color='black', back_color='white')
output = BASE / 'assets' / 'diagrams' / 'certificate_verification_qr.png'
img.save(output)
plt.figure(figsize=(4, 4))
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.title('Пример QR-кода проверки')
plt.show()
output

## 3. Публичный ответ проверки

Публичный ответ должен быть достаточным для проверки подлинности, но не должен превращаться в источник избыточных персональных данных.

In [ ]:
payload_path = BASE / 'data' / 'example_certificate_payload.json'
with payload_path.open(encoding='utf-8') as f:
    payload = json.load(f)
public_response = payload['publicVerificationResponse']
public_response

In [ ]:
public_fields = pd.DataFrame([
    {'field': k, 'example': v, 'purpose': 'публичная проверка подлинности'}
    for k, v in public_response.items()
])
public_fields

## 4. Поля, которые нельзя раскрывать на публичной странице

Публичная страница проверки не должна показывать внутренние идентификаторы, email, телефон, ключи хранения файлов, хеши и служебные метаданные.

In [ ]:
pd.DataFrame({'non_public_field': payload['privateFieldsNotForPublicPage']})

## 5. Схема процесса проверки

Ниже процесс представлен как последовательность действий: сканирование QR-кода, переход по ссылке, поиск записи по `verificationCode`, проверка статуса и возврат безопасного ответа.

In [ ]:
import networkx as nx
steps = [
    ('QR on certificate', 'verificationUrl'),
    ('verificationUrl', 'GET /certificates/verify/{code}'),
    ('GET /certificates/verify/{code}', 'IssuedCertificate lookup'),
    ('IssuedCertificate lookup', 'status check'),
    ('status check', 'public response'),
]
G = nx.DiGraph()
G.add_edges_from(steps)
plt.figure(figsize=(13, 4))
pos = {node: (i, 0) for i, node in enumerate(G.nodes())}
nx.draw_networkx_nodes(G, pos, node_size=2600)
nx.draw_networkx_labels(G, pos, font_size=9)
nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle='-|>', arrowsize=18)
plt.title('Поток публичной проверки сертификата')
plt.axis('off')
output = BASE / 'assets' / 'diagrams' / 'qr_verification_flow.png'
plt.savefig(output, dpi=200, bbox_inches='tight')
plt.show()
output

## 6. Вывод

QR-проверка должна быть публичной, но ограниченной. Она подтверждает факт выдачи и статус документа, не раскрывая внутренние данные пользователя и не предоставляя доступ к закрытым материалам курса.